In [0]:
# Imports
import requests
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import lit, current_timestamp

print("Imports successful")

In [0]:
# Define indicators and countries


COUNTRIES  = "KE;UG;TZ;ET;GH;NG;ZA;SN;ML;MZ;RW;ZM;ZW;CM;CI"
BASE_URL   = "https://api.worldbank.org/v2/country"

INDICATORS = [
    {
        "code": "SN.ITK.DEFC.ZS",
        "name": "undernourishment_pct"
    },
    {
        "code": "SH.STA.STNT.ZS",
        "name": "stunting_pct"
    },
    {
        "code": "SH.STA.MALN.ZS",
        "name": "underweight_pct"
    },
]

print(f"Targeting {len(INDICATORS)} food security indicators")
print(f"Across {len(COUNTRIES.split(';'))} countries")

In [0]:
# Fetch function

def fetch_world_bank(indicator_code: str, countries: str) -> list:
    """
    Fetches a single World Bank indicator for all target countries.
    Returns raw list of records.
    """
    url = f"{BASE_URL}/{countries}/indicator/{indicator_code}"
    params = {
        "format":   "json",
        "per_page": 500,
        "date":     "2010:2023",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data    = response.json()
    records = data[1] if data and len(data) > 1 else []
    return records

# Test on first indicator
test = fetch_world_bank(INDICATORS[0]["code"], COUNTRIES)
print(f"Test fetch returned {len(test)} records")
print(f"Sample: {test[0]}")

In [0]:
# Cell 4 — flatten function
# Same structure as notebook 01 but we add indicator_name
# so we can tell records apart when all three land in one table

def flatten_records(records: list, indicator_name: str) -> list:
    """
    Flattens World Bank API response into row-level dicts.
    Skips nulls — not all countries report all indicators every year.
    """
    flat = []
    for r in records:
        if r.get("value") is None:
            continue
        flat.append({
            "country_code":   r["country"]["id"],
            "country_name":   r["country"]["value"],
            "country_iso3":   r.get("countryiso3code", ""),
            "indicator_id":   r["indicator"]["id"],
            "indicator_name": indicator_name,
            "year":           int(r["date"]),
            "value":          float(r["value"]),
        })
    return flat

# Test flatten
flat_test = flatten_records(test, INDICATORS[0]["name"])
print(f"Flattened: {len(flat_test)} non-null records")
print(f"Sample: {flat_test[0]}")

In [0]:
# Fetch all three indicators and combine
# Why combine into one table?
# In Silver we'll pivot this into columns per country per year
# Having all indicators in one Bronze table with an indicator_name
# column is the standard "long format" — easier to extend later

import time

all_flat = []

for ind in INDICATORS:
    print(f"Fetching: {ind['name']}...", end=" ")
    try:
        raw   = fetch_world_bank(ind["code"], COUNTRIES)
        flat  = flatten_records(raw, ind["name"])
        all_flat.extend(flat)
        print(f"{len(flat)} records")
    except Exception as e:
        print(f"FAILED — {e}")
    time.sleep(2)  # be polite to the API

print(f"\nTotal combined records: {len(all_flat)}")

In [0]:
# Create Spark DataFrame with explicit schema

schema = StructType([
    StructField("country_code",   StringType(),  True),
    StructField("country_name",   StringType(),  True),
    StructField("country_iso3",   StringType(),  True),
    StructField("indicator_id",   StringType(),  True),
    StructField("indicator_name", StringType(),  True),
    StructField("year",           IntegerType(), True),
    StructField("value",          DoubleType(),  True),
])

df_food = spark.createDataFrame(all_flat, schema=schema)

df_food = (df_food
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source",      lit("world_bank_api"))
    .withColumn("domain",      lit("food_security"))
)

print(f"DataFrame: {df_food.count()} rows, {len(df_food.columns)} columns")
df_food.printSchema()

In [0]:
# Write to Bronze Delta table

BRONZE_TABLE = "workspace.default.bronze_world_bank_food"

(df_food.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Written to Bronze: {BRONZE_TABLE}")

In [0]:
# Verify and inspect

BRONZE_TABLE = "workspace.default.bronze_world_bank_food"
df_check = spark.read.table(BRONZE_TABLE)

print(f"Total rows: {df_check.count()}")
print(f"\nRecords per indicator:")
display(
    df_check.groupBy("indicator_name")
            .count()
            .orderBy("indicator_name")
)

In [0]:
# Check value ranges look sensible
# Undernourishment in Sub-Saharan Africa typically ranges 5–60%

from pyspark.sql.functions import round as spark_round, avg, min, max

print("Value ranges per indicator:")
display(
    df_check.groupBy("indicator_name")
            .agg(
                spark_round(avg("value"), 2).alias("avg"),
                spark_round(min("value"), 2).alias("min"),
                spark_round(max("value"), 2).alias("max"),
            )
            .orderBy("indicator_name")
)